# 7. Self-Correction Loop

The **SelfCorrectionLoop** wraps the DAG executor and critic. If the critic rejects any outputs, the loop re-runs just the failed sub-tasks (up to `max_retries` times).

This makes the system resilient to transient LLM failures and hallucination.

In [ ]:
from unittest.mock import AsyncMock, MagicMock
from mares.orchestrator.self_correction_loop import SelfCorrectionLoop
from mares.models.task import Task
from mares.models.sub_task import SubTask
from mares.models.agent_output import AgentOutput

# Mock the DAG executor and critic
mock_dag = MagicMock()
mock_critic = MagicMock()

# First call returns failed outputs, second call returns passing
call_count = [0]
async def mock_dag_run(task):
    return [
        AgentOutput(agent="research_agent", sub_task_id=1, content="ok"),
        AgentOutput(agent="execution_agent", sub_task_id=2, content="ok", metadata={"success": True}),
    ]

mock_dag.run = mock_dag_run

async def mock_critic_run(outputs):
    call_count[0] += 1
    from mares.models.critic_report import CriticReport
    if call_count[0] == 1:
        # First run fails
        return CriticReport(
            passed=False,
            issues=[{"sub_task_id": 1, "severity": "high", "description": "Missing source"}],
            summary="Retry needed"
        )
    return CriticReport(passed=True, issues=[], summary="All good")

mock_critic.run = mock_critic_run

loop = SelfCorrectionLoop(dag_executor=mock_dag, critic=mock_critic, max_retries=2)
task = Task(description="Test", sub_tasks=[
    SubTask(id=1, task="Research topic", depends_on=[]),
    SubTask(id=2, task="Write code", depends_on=[1]),
])

outputs = await loop.run(task)
print(f"Final outputs: {len(outputs)}")
print(f"Critic called {call_count[0]} times")

## Retry Strategy

The self-correction loop:
1. Executes the full DAG
2. Runs the critic on all outputs
3. If failed, identifies which sub-task IDs failed
4. Re-runs only those sub-tasks (not the entire DAG)
5. Merges new results with the passing ones
6. Repeats up to `max_retries` times

In [ ]:
# Show the retry configuration
print(f"Default max_retries: {loop.max_retries}")

# Simulate exhaustion of retries
call_count2 = [0]
async def always_fail(outputs):
    call_count2[0] += 1
    from mares.models.critic_report import CriticReport
    return CriticReport(passed=False, issues=[], summary="Always fails")

mock_critic2 = MagicMock()
mock_critic2.run = always_fail

loop2 = SelfCorrectionLoop(dag_executor=mock_dag, critic=mock_critic2, max_retries=3)
outputs2 = await loop2.run(task)
print(f"Critic called {call_count2[0]} times (should be 4 = 3 retries + 1 final)")
print(f"Returns all outputs even when correction is exhausted")